# Silver Layer: Data Integration and Null Handling

This notebook performs data integration (joining Retention and BOB datasets). It ensures the output contains only unique account numbers by grouping by account number. During the grouping:
- Total, Lost, and Active combinations are tracked to categorize **Churn**.
- All other numerical features are averaged (`mean`).
- All other categorical features use the most frequent value (`mode`).

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [5]:
# Load cleaned datasets
date_cols_retention = ['customer_since', 'agreement_end_date', 'case_creation_date', 'resolved_date', 'registered_date', 'expected_pull_date']
date_cols_bob = ['agreement_start_date', 'agreement_end_date']

retention_df = pd.read_csv('../../data/02_intermediate/cleaned_retention.csv', parse_dates=date_cols_retention)
bob_df = pd.read_csv('../../data/02_intermediate/cleaned_bob.csv', parse_dates=date_cols_bob)

print(f"Retention shape: {retention_df.shape}")
print(f"BOB shape: {bob_df.shape}")

# Drop rows with null join keys
retention_df = retention_df.dropna(subset=['customer_account_number'])
bob_df = bob_df.dropna(subset=['account_number'])

print(f"After dropping null keys - Retention shape: {retention_df.shape}")
print(f"After dropping null keys - BOB shape: {bob_df.shape}")

Retention shape: (85411, 29)
BOB shape: (376803, 26)
After dropping null keys - Retention shape: (77949, 29)
After dropping null keys - BOB shape: (376784, 26)


In [6]:
def label_churn(row):
    active = row['active_agreements']
    lost = row['lost_agreements']
    if active == 0:
        return 2  # Full Churn
    elif lost == 0:
        return 0  # No Churn
    return 1      # Partial Churn

def get_first_mode(x):
    m = x.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

def join_and_aggregate(retention_df, bob_df):
    retention_unwanted = [
        'case_id', 'country', 'customer_name', 'customer_since',
        'registered_time', 'resolved_time'
    ]
    bob_unwanted = [
        'postal_code', 'vat_number', 'system_status', 'is_bob', 'bpg',
        'msdyn_product_number', 'product_name', 'machine', 'machine_variant',
        'chemistry'
    ]

    retention_clean = retention_df.drop(columns=[col for col in retention_unwanted if col in retention_df.columns])
    bob_clean = bob_df.drop(columns=[col for col in bob_unwanted if col in bob_df.columns])

    # --- 0. Aggregate BOB dataset before any joins ---
    numeric_cols_bob = []
    categorical_cols_bob = []
    for col in bob_clean.columns:
        if col == 'account_number':
            continue
        if pd.api.types.is_numeric_dtype(bob_clean[col]):
            numeric_cols_bob.append(col)
        else:
            categorical_cols_bob.append(col)

    agg_funcs_bob = {}
    for col in numeric_cols_bob:
        agg_funcs_bob[col] = 'mean'
    for col in categorical_cols_bob:
        agg_funcs_bob[col] = get_first_mode

    print("Aggregating BOB dataset...")
    cleaned_bob = bob_clean.groupby('account_number', dropna=False, as_index=False).agg(agg_funcs_bob)

    joined_df = pd.merge(
        retention_clean,
        cleaned_bob,
        left_on='customer_account_number',
        right_on='account_number',
        how='left',
        suffixes=('_retention', '_bob')
    )

    # Preserve the original account key
    joined_df['account_number'] = joined_df['account_number'].fillna(joined_df['customer_account_number'])
    joined_df = joined_df.drop(columns=['customer_account_number'], errors='ignore')
    joined_df = joined_df.dropna(subset=['account_number'])

    # Resolve dual columns
    for col in list(joined_df.columns):
        if col.endswith('_bob'):
            base = col[:-4]
            dual = f"{base}_retention"
            if dual in joined_df.columns:
                joined_df = joined_df.drop(columns=[dual])
                joined_df = joined_df.rename(columns={col: base})
        elif col.endswith('_retention'):
            base = col[:-10]
            if base not in joined_df.columns:
                joined_df = joined_df.rename(columns={col: base})

    print(f"Data shape before groupby: {joined_df.shape}")

    # --- 1. Compute Agreement Targets specifically ---
    agg_targets = joined_df.groupby('account_number', dropna=False).agg(
        total_agreements=('agreement_number', 'nunique')
    ).reset_index()
    
    lost_df = joined_df[joined_df['resolution_status'] == 'Customer Lost']
    lost_counts = lost_df.groupby('account_number', dropna=False)['agreement_number'].nunique().reset_index()
    lost_counts = lost_counts.rename(columns={'agreement_number': 'lost_agreements'})
    agg_targets = agg_targets.merge(lost_counts, on='account_number', how='left')
    agg_targets['lost_agreements'] = agg_targets['lost_agreements'].fillna(0).astype(int)

    active_df = joined_df[joined_df['resolution_status'] != 'Customer Lost']
    active_counts = active_df.groupby('account_number', dropna=False)['agreement_number'].nunique().reset_index()
    active_counts = active_counts.rename(columns={'agreement_number': 'active_agreements'})
    agg_targets = agg_targets.merge(active_counts, on='account_number', how='left')
    agg_targets['active_agreements'] = agg_targets['active_agreements'].fillna(0).astype(int)

    agg_targets['churn_category'] = agg_targets.apply(label_churn, axis=1)

    # --- 2. Compute Mean & Mode for all other columns ---
    numeric_cols = []
    categorical_cols = []
    for col in joined_df.columns:
        if col in ['account_number', 'agreement_number', 'resolution_status']:
            continue
        if pd.api.types.is_numeric_dtype(joined_df[col]):
            numeric_cols.append(col)
        else:
            categorical_cols.append(col)

    agg_funcs = {}
    for col in numeric_cols:
        agg_funcs[col] = 'mean'
    
    for col in categorical_cols:
        agg_funcs[col] = get_first_mode

    print("Grouping by account_number...")
    grouped_df = joined_df.groupby('account_number', as_index=False).agg(agg_funcs)
    
    # --- 3. Merge targets back ---
    final_df = grouped_df.merge(agg_targets, on='account_number', how='left')
    
    return final_df

def global_fill_missing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if col == 'account_number':
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            mean_val = df[col].mean(skipna=True)
            if pd.notna(mean_val):
                df[col] = df[col].fillna(mean_val)
            else:
                df[col] = df[col].fillna(0)
        else:
            mode_val = df[col].mode(dropna=True)
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val.iloc[0])
            else:
                df[col] = df[col].fillna('Unknown')
    return df

# Run the integration
joined_df = join_and_aggregate(retention_df, bob_df)
joined_df = global_fill_missing(joined_df)

print(f"Joined uniqueness account dataset shape: {joined_df.shape}")

print("\n Account-level churn distribution:")
print(joined_df['churn_category'].value_counts())

# Save to silver
os.makedirs('../../data/02_intermediate', exist_ok=True)
joined_df.to_csv('../../data/02_intermediate/joined_data.csv', index=False)
print("\n Saved uniquely grouped joined dataset to ../../data/02_intermediate/joined_data.csv")

Aggregating BOB dataset...
Data shape before groupby: (77949, 38)
Grouping by account_number...
Joined uniqueness account dataset shape: (21526, 40)

 Account-level churn distribution:
churn_category
2    16233
0     4226
1     1067
Name: count, dtype: int64

 Saved uniquely grouped joined dataset to ../../data/02_intermediate/joined_data.csv
